In [ ]:
import numpy as np
from scipy.interpolate import LinearNDInterpolator, RegularGridInterpolator
import spacepy.pybats.bats as pybats
import pathlib
import glob
import re
import joblib
from tqdm import tqdm
from datetime import datetime
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

# =============================================================================
# 1. SETUP & TIME EXTRACTION (Unchanged)
# =============================================================================
RE = 6371000.0 
data_dir = "/Users/nfurioso/Library/CloudStorage/OneDrive-UniversityofFlorida/Research/Journal_Paper/GC_motion_in_MHD/MHDoutputValues/BATSRUS/"
search_path = str(pathlib.Path(data_dir) / "3d__var_1_e*.out")

file_paths = sorted(glob.glob(search_path))
if not file_paths:
    raise FileNotFoundError(f"No files found matching {search_path}")

times_sec = []
for f in file_paths:
    match = re.search(r'e(\d{8})-(\d{6})-', f)
    if match:
        dt_obj = datetime.strptime(match.group(1) + match.group(2), "%Y%m%d%H%M%S")
        times_sec.append(dt_obj.timestamp())
    else:
        raise ValueError(f"Could not parse time from filename: {f}")

times_sec = np.array(times_sec)
times_sec -= times_sec[0] 

# =============================================================================
# 2. DEFINE THE GRID (Unchanged)
# =============================================================================
def make_axis_re(min_re, max_re, dense_bounds_re, res_dense_re, res_coarse_re):
    d_min, d_max = dense_bounds_re
    core = np.arange(d_min, d_max, res_dense_re, dtype=np.float32)
    left = np.arange(min_re, d_min, res_coarse_re, dtype=np.float32)
    right = np.arange(d_max, max_re, res_coarse_re, dtype=np.float32)
    return np.unique(np.concatenate([left, core, right]))

x_re = make_axis_re(-20, 20, (-10, 10), 0.1, 0.5)
y_re = make_axis_re(-20, 20, (-10, 10), 0.1, 0.5)
z_re = make_axis_re(-20, 20, (-10, 10), 0.1, 0.5)

nx, ny, nz = len(x_re), len(y_re), len(z_re)
nt = len(times_sec)

# Full Volume 4D Grids
Bx_4d, By_4d, Bz_4d = np.zeros((nt, nx, ny, nz), dtype=np.float32), np.zeros((nt, nx, ny, nz), dtype=np.float32), np.zeros((nt, nx, ny, nz), dtype=np.float32)
Vx_4d, Vy_4d, Vz_4d = np.zeros((nt, nx, ny, nz), dtype=np.float32), np.zeros((nt, nx, ny, nz), dtype=np.float32), np.zeros((nt, nx, ny, nz), dtype=np.float32)

# --- NEW: Full 4D Grids for Density and Temperature ---
n_4d = np.zeros((nt, nx, ny, nz), dtype=np.float32)
kT_4d = np.zeros((nt, nx, ny, nz), dtype=np.float32)

# =============================================================================
# 3. LOOP THROUGH FILES & BAKE
# =============================================================================
num_chunks = 4
x_splits = np.array_split(np.arange(nx), num_chunks)
y_splits = np.array_split(np.arange(ny), num_chunks)
z_splits = np.array_split(np.arange(nz), num_chunks)
total_chunks = num_chunks**3

for t_idx, path in enumerate(file_paths):
    print(f"\n--- File {t_idx+1}/{nt} ---")
    BATSRUS = pybats.IdlFile(path)

    raw_coords = np.column_stack([BATSRUS['x'], BATSRUS['y'], BATSRUS['z']]).astype(np.float32)
    # --- ADDED: rho and p to the data extraction ---
    raw_data = np.column_stack([BATSRUS['bx'], BATSRUS['by'], BATSRUS['bz'], 
                                BATSRUS['ux'], BATSRUS['uy'], BATSRUS['uz'],
                                BATSRUS['rho'], BATSRUS['p']]).astype(np.float32)

    buff = 1.0
    mask = ((raw_coords[:,0] >= x_re.min() - buff) & (raw_coords[:,0] <= x_re.max() + buff) &
            (raw_coords[:,1] >= y_re.min() - buff) & (raw_coords[:,1] <= y_re.max() + buff) &
            (raw_coords[:,2] >= z_re.min() - buff) & (raw_coords[:,2] <= z_re.max() + buff))

    subset_coords = raw_coords[mask]
    subset_data = raw_data[mask]

    # --- Existing Volume Chunking for B, V, n, and kT ---
    with tqdm(total=total_chunks, unit="chunk", desc=f"t={times_sec[t_idx]}s") as pbar:
        for x_idx in x_splits:
            for y_idx in y_splits:
                for z_idx in z_splits:
                    # Inside your x, y, z chunking loops:
                    cx, cy, cz = x_re[x_idx], y_re[y_idx], z_re[z_idx]

                    # Calculate the approximate radial distance of this chunk's center
                    r_center = np.sqrt(np.mean(cx)**2 + np.mean(cy)**2 + np.mean(cz)**2)

                    # Dynamic buffer: 1.0 Re near Earth, expanding to 4.0 Re in the distant tail
                    dynamic_buff = 1.0 if r_center < 10.0 else 4.0

                    c_mask = ((subset_coords[:,0] >= cx.min() - dynamic_buff) & (subset_coords[:,0] <= cx.max() + dynamic_buff) &
                            (subset_coords[:,1] >= cy.min() - dynamic_buff) & (subset_coords[:,1] <= cy.max() + dynamic_buff) &
                            (subset_coords[:,2] >= cz.min() - dynamic_buff) & (subset_coords[:,2] <= cz.max() + dynamic_buff))
                    
                    # NOTE: Interpolating 8 columns now (B, V, rho, p)
                    local_interp = LinearNDInterpolator(subset_coords[c_mask], subset_data[c_mask, 0:8], fill_value=0.0)
                    L_XX, L_YY, L_ZZ = np.meshgrid(cx, cy, cz, indexing='ij')
                    
                    # Reshape to include all 8 variables
                    local_res = local_interp(np.column_stack([L_XX.ravel(), L_YY.ravel(), L_ZZ.ravel()])).reshape(len(cx), len(cy), len(cz), 8)
                    
                    Bx_4d[t_idx, x_idx[0]:x_idx[-1]+1, y_idx[0]:y_idx[-1]+1, z_idx[0]:z_idx[-1]+1] = local_res[..., 0]
                    By_4d[t_idx, x_idx[0]:x_idx[-1]+1, y_idx[0]:y_idx[-1]+1, z_idx[0]:z_idx[-1]+1] = local_res[..., 1]
                    Bz_4d[t_idx, x_idx[0]:x_idx[-1]+1, y_idx[0]:y_idx[-1]+1, z_idx[0]:z_idx[-1]+1] = local_res[..., 2]
                    Vx_4d[t_idx, x_idx[0]:x_idx[-1]+1, y_idx[0]:y_idx[-1]+1, z_idx[0]:z_idx[-1]+1] = local_res[..., 3]
                    Vy_4d[t_idx, x_idx[0]:x_idx[-1]+1, y_idx[0]:y_idx[-1]+1, z_idx[0]:z_idx[-1]+1] = local_res[..., 4]
                    Vz_4d[t_idx, x_idx[0]:x_idx[-1]+1, y_idx[0]:y_idx[-1]+1, z_idx[0]:z_idx[-1]+1] = local_res[..., 5]
                    
                    # --- NEW: Process Density and Temperature for this chunk ---
                    rho_local = local_res[..., 6]
                    p_local = local_res[..., 7]
                    
                    # Conversion: Mp/cc -> m^-3 and nPa -> N/m^2
                    n_local = rho_local * 1e6
                    P_pa_local = p_local * 1e-9
                    n_safe = np.maximum(n_local, 1e-10)
                    
                    n_4d[t_idx, x_idx[0]:x_idx[-1]+1, y_idx[0]:y_idx[-1]+1, z_idx[0]:z_idx[-1]+1] = n_local
                    kT_4d[t_idx, x_idx[0]:x_idx[-1]+1, y_idx[0]:y_idx[-1]+1, z_idx[0]:z_idx[-1]+1] = (P_pa_local / n_safe)
                    
                    pbar.update(1)

# =============================================================================
# 4. SMOOTHING & COMPILE (Unchanged for B/V, Appended for n/kT)
# =============================================================================

# Use axes in meters for physical derivatives
x_m = (x_re * RE).astype(np.float32)
y_m = (y_re * RE).astype(np.float32)
z_m = (z_re * RE).astype(np.float32)

print("\nCalculating Derivatives and Applying Selective Smoothing...")

# sigma = (time, X, Y, Z)
sigma_tuple = (0, 0.5, 0.5, 0.5) 
Bx_smooth = gaussian_filter(Bx_4d, sigma=sigma_tuple)
By_smooth = gaussian_filter(By_4d, sigma=sigma_tuple)
Bz_smooth = gaussian_filter(Bz_4d, sigma=sigma_tuple)

# Calculate B magnitude and unit vectors using the SMOOTHED arrays
Bmag_smooth = np.sqrt(Bx_smooth**2 + By_smooth**2 + Bz_smooth**2)
Bmag_smooth = np.maximum(Bmag_smooth, np.float32(1e-15))

bx_hat = Bx_smooth / Bmag_smooth
by_hat = By_smooth / Bmag_smooth
bz_hat = Bz_smooth / Bmag_smooth

# Now run your np.gradient calculations on these smoothed versions
gradB_x, gradB_y, gradB_z = np.gradient(Bmag_smooth, x_m, y_m, z_m, axis=(1, 2, 3))

# Cast to float32 to save RAM before the next step
gradB_x = gradB_x.astype(np.float32)
gradB_y = gradB_y.astype(np.float32)
gradB_z = gradB_z.astype(np.float32)

# 2. Spatial derivatives of unit vectors
print("  >> Calculating curvature components...")
dbx_dx, dbx_dy, dbx_dz = np.gradient(bx_hat, x_m, y_m, z_m, axis=(1, 2, 3))
dby_dx, dby_dy, dby_dz = np.gradient(by_hat, x_m, y_m, z_m, axis=(1, 2, 3))
dbz_dx, dbz_dy, dbz_dz = np.gradient(bz_hat, x_m, y_m, z_m, axis=(1, 2, 3))

# 3. Curvature vector: kappa = (b_hat dot nabla) b_hat
kappa_x = (bx_hat * dbx_dx + by_hat * dbx_dy + bz_hat * dbx_dz).astype(np.float32)
kappa_y = (bx_hat * dby_dx + by_hat * dby_dy + bz_hat * dby_dz).astype(np.float32)
kappa_z = (bx_hat * dbz_dx + by_hat * dbz_dy + bz_hat * dbz_dz).astype(np.float32)

del dbx_dx, dbx_dy, dbx_dz, dby_dx, dby_dy, dby_dz, dbz_dx, dbz_dy, dbz_dz, bx_hat, by_hat, bz_hat
del Bx_smooth, By_smooth, Bz_smooth, Bmag_smooth # <-- Added cleanup

# =============================================================================
# 5. COMPILE & SAVE 4D INTERPOLATORS
# =============================================================================
print("\nBuilding Final 4D Interpolators...")

x_meters = x_re * RE
y_meters = y_re * RE
z_meters = z_re * RE

# Check if you have enough time steps for cubic interpolation
interp_method = 'cubic' if nt >= 4 else 'linear'

kwargs = {'bounds_error': False, 'fill_value': 0.0, 'method': interp_method}

output_data = {
    'Bx': RegularGridInterpolator((times_sec, x_meters, y_meters, z_meters), Bx_4d, **kwargs),
    'By': RegularGridInterpolator((times_sec, x_meters, y_meters, z_meters), By_4d, **kwargs),
    'Bz': RegularGridInterpolator((times_sec, x_meters, y_meters, z_meters), Bz_4d, **kwargs),
    'Vx': RegularGridInterpolator((times_sec, x_meters, y_meters, z_meters), Vx_4d, **kwargs),
    'Vy': RegularGridInterpolator((times_sec, x_meters, y_meters, z_meters), Vy_4d, **kwargs),
    'Vz': RegularGridInterpolator((times_sec, x_meters, y_meters, z_meters), Vz_4d, **kwargs),
    'gradBx': RegularGridInterpolator((times_sec, x_meters, y_meters, z_meters), gradB_x, **kwargs),
    'gradBy': RegularGridInterpolator((times_sec, x_meters, y_meters, z_meters), gradB_y, **kwargs),
    'gradBz': RegularGridInterpolator((times_sec, x_meters, y_meters, z_meters), gradB_z, **kwargs),
    'kappaX': RegularGridInterpolator((times_sec, x_meters, y_meters, z_meters), kappa_x, **kwargs),
    'kappaY': RegularGridInterpolator((times_sec, x_meters, y_meters, z_meters), kappa_y, **kwargs),
    'kappaZ': RegularGridInterpolator((times_sec, x_meters, y_meters, z_meters), kappa_z, **kwargs),
    
    # --- NEW: Full 4D Scalar Interpolators ---
    'n_bnd': RegularGridInterpolator((times_sec, x_meters, y_meters, z_meters), n_4d, **kwargs),
    'kT_bnd': RegularGridInterpolator((times_sec, x_meters, y_meters, z_meters), kT_4d, **kwargs),
    
    'times_sec': times_sec,
    'units': {
        'B_fields': 'nT', 'V_fields': 'km/s', 'gradB': 'nT/m', 'kappa': '1/m',
        'n_bnd': 'm^-3', 'kT_bnd': 'Joules', 'time': 'Seconds', 'grid': 'Meters'
    }
}

joblib.dump(output_data, "Interpolators_4D_Float32.pkl", compress=3)
print("Complete. Boundary Kappa moments included.")

# =============================================================================
# 5. VISUALIZATION / VALIDATION PLOTS
# =============================================================================
print("\nGenerating Validation Plots for the Equatorial Plane (Z=0)...")

# Choose the Z=0 plane. We pass the axes in meters to match the interpolator's expectations.
z_plot = 0.0
X_plot, Y_plot = np.meshgrid(x_meters, y_meters, indexing='ij')

# Build the list of times: Actual files + halfway points between all files
plot_times = []
time_labels = []

for i in range(len(times_sec)):
    # Add the actual file time
    plot_times.append(times_sec[i])
    time_labels.append(f"File {i+1}")
    
    # If it is not the last file, add the halfway point to the next file
    if i < len(times_sec) - 1:
        t_mid = (times_sec[i] + times_sec[i+1]) / 2.0
        plot_times.append(t_mid)
        time_labels.append(f"Halfway {i+1}-{i+2}")

num_cols = len(plot_times)
print(f"Generating {num_cols} time slices for validation...")

# Set up the plot window (2 rows for B and V). 
# We dynamically scale the figure width (4 inches per column) so it doesn't get squished.
fig, axes = plt.subplots(2, num_cols, figsize=(4 * num_cols, 8))
if num_cols == 1:
    axes = np.expand_dims(axes, axis=1)

for i, t in enumerate(plot_times):
    print(f"  Plotting slice {i+1}/{num_cols} for t={t}s...")
    
    # Query the 4D interpolator directly from the dictionary
    bx = output_data['Bx']((t, X_plot, Y_plot, z_plot))
    by = output_data['By']((t, X_plot, Y_plot, z_plot))
    bz = output_data['Bz']((t, X_plot, Y_plot, z_plot))
    
    vx = output_data['Vx']((t, X_plot, Y_plot, z_plot))
    vy = output_data['Vy']((t, X_plot, Y_plot, z_plot))
    vz = output_data['Vz']((t, X_plot, Y_plot, z_plot))
    
    # Calculate magnitudes
    b_mag = np.sqrt(bx**2 + by**2 + bz**2)
    v_mag = np.sqrt(vx**2 + vy**2 + vz**2)
    
    # Convert X/Y back to Earth Radii for readable plot axes
    X_re_plot = X_plot / RE
    Y_re_plot = Y_plot / RE

    # --- Plot Magnetic Field Magnitude ---
    ax_b = axes[0, i]
    # Using log scale color mapping for B-field since it varies drastically near Earth
    im_b = ax_b.pcolormesh(X_re_plot, Y_re_plot, b_mag, cmap='viridis', shading='auto', 
                           norm=plt.matplotlib.colors.LogNorm(vmin=1, vmax=np.nanmax(b_mag)))
    ax_b.set_title(f"|B| (nT)\n{time_labels[i]} (t={t}s)")
    ax_b.set_xlabel("X (Re)")
    ax_b.set_ylabel("Y (Re)")
    ax_b.set_aspect('equal')
    
    # Earth outline for reference
    circle1 = plt.Circle((0, 0), 1, color='w', fill=False, linestyle='--')
    ax_b.add_patch(circle1)
    fig.colorbar(im_b, ax=ax_b, fraction=0.046, pad=0.04)

    # --- Plot Velocity Field Magnitude ---
    ax_v = axes[1, i]
    im_v = ax_v.pcolormesh(X_re_plot, Y_re_plot, v_mag, cmap='plasma', shading='auto')
    ax_v.set_title(f"|V| (km/s)\n{time_labels[i]} (t={t}s)")
    ax_v.set_xlabel("X (Re)")
    ax_v.set_ylabel("Y (Re)")
    ax_v.set_aspect('equal')
    
    # Earth outline for reference
    circle2 = plt.Circle((0, 0), 1, color='w', fill=False, linestyle='--')
    ax_v.add_patch(circle2)
    fig.colorbar(im_v, ax=ax_v, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig("Interpolation_Validation.png", dpi=300, bbox_inches='tight')
print("Plots saved to 'Interpolation_Validation.png'. Displaying now...")
plt.show()

/Users/nfurioso/mhd_env/lib/python3.11/site-packages/spacepy/empiricals.py:26: UserWarning: Qin-Denton/OMNI2 data not found in current format. This module has limited functionality. Run spacepy.toolbox.update(QDomni=True) to download data.
  import spacepy.omni as om



--- File 1/5 ---


t=0.0s:   0%|          | 0/64 [00:00<?, ?chunk/s]